In [ ]:
import time
import chromadb
from sentence_transformers import SentenceTransformer
import pandas as pd
import os
from dotenv import load_dotenv
load_dotenv()

In [ ]:
# --- Setup ---
model  = SentenceTransformer("all-MiniLM-L6-v2")
client = chromadb.PersistentClient(path="./chroma_db")

In [ ]:
def reset_collection(client, collection_name):
    try:
        client.delete_collection(collection_name)
    except Exception:
        pass  # Collection didn't exist yet
    return client.create_collection(name=collection_name, metadata={"hnsw:space": "cosine"})

COLLECTION_01=os.getenv("COLLECTION_01")
COLLECTION_02=os.getenv("COLLECTION_02")
COLLECTION_03=os.getenv("COLLECTION_03")
COLLECTION_04=os.getenv("COLLECTION_04")
COLLECTION_05=os.getenv("COLLECTION_05")
COLLECTION_06=os.getenv("COLLECTION_06")
COLLECTION_07=os.getenv("COLLECTION_07")
COLLECTION_08=os.getenv("COLLECTION_08")
COLLECTION_09=os.getenv("COLLECTION_09")
COLLECTION_10=os.getenv("COLLECTION_10")
COLLECTION_11=os.getenv("COLLECTION_11")

collections = {
    # COLLECTION_01:                  reset_collection(client, COLLECTION_01),
    # COLLECTION_02:            reset_collection(client, COLLECTION_02),
    # COLLECTION_03:       reset_collection(client, COLLECTION_03),
    # COLLECTION_04:            reset_collection(client, COLLECTION_04),
    # COLLECTION_05:                 reset_collection(client, COLLECTION_05),
    # COLLECTION_06:               reset_collection(client, COLLECTION_06),
    COLLECTION_07:           reset_collection(client, COLLECTION_07),
    # COLLECTION_08:   reset_collection(client, COLLECTION_08),
    # COLLECTION_09:         reset_collection(client, COLLECTION_09),
    # COLLECTION_10:   reset_collection(client, COLLECTION_10),
    # COLLECTION_11:  reset_collection(client, COLLECTION_11),
}

COLLECTION_WEIGHTS = {
    # COLLECTION_01:                  1.0,
    # COLLECTION_02:            1.0,
    # COLLECTION_03:       1.0,
    # COLLECTION_04:            1.0,
    # COLLECTION_05:                 1.0,
    # COLLECTION_06:               1.0,
    COLLECTION_07:           1.0,
    # COLLECTION_08:   1.0,
    # COLLECTION_09:         1.0,
    # COLLECTION_10:     1.0,
    # COLLECTION_11:  1.0,
}

# --- Loaders ---
def load_chunks_csv(filepath: str) -> list[dict]:
    df = pd.read_csv(filepath)
    chunks = []
    for _, row in df.iterrows():
        metadata = json.loads(row["metadata"])
        metadata["product_id"] = str(metadata["product_id"])   # normalize once, here
        chunks.append({
            "metadata": metadata,
            "text":     row["text"]
        })
    print(f"Loaded {len(chunks)} chunks from {filepath}")
    return chunks

# --- Store chunks ---
def embed_and_store_chunks(chunks: list[dict], collection, doc_type: str, batch_size: int = 100):
    items = [c for c in chunks if c.get("text") and str(c["text"]).strip()]
    print(f"Storing {len(items)} chunks in '{doc_type}'...")

    encode_time = 0.0
    upsert_time = 0.0

    for i in range(0, len(items), batch_size):
        batch = items[i:i + batch_size]
        texts = [c["text"] for c in batch]

        t0 = time.perf_counter()
        embeddings = model.encode(texts, show_progress_bar=False).tolist()
        encode_time += time.perf_counter() - t0

        metadatas = [{"doc_type": doc_type, **c["metadata"]} for c in batch]
        ids = [f"{doc_type}_{c['metadata']['product_id']}_{i + j}" for j, c in enumerate(batch)]

        t1 = time.perf_counter()
        collection.upsert(ids=ids, embeddings=embeddings, documents=texts, metadatas=metadatas)
        upsert_time += time.perf_counter() - t1

    total = encode_time + upsert_time
    throughput = len(items) / encode_time if encode_time > 0 else float("nan")
    print(f"Done storing '{doc_type}': {len(items)} chunks")
    print(f"  Encoding time: {encode_time:.2f}s ({throughput:.1f} records/sec)")
    print(f"  Upsert time:   {upsert_time:.2f}s")
    print(f"  Total:         {total:.2f}s")

    return {"doc_type": doc_type, "n": len(items), "encode_s": encode_time, "upsert_s": upsert_time}


# --- Load all CSVs ---
# COLLECTION_01_CHUNKS                  = load_chunks_csv("processed_chunks/COLLECTION_01_CHUNKS.csv")
# COLLECTION_02_CHUNKS             = load_chunks_csv("processed_chunks/COLLECTION_02_CHUNKS.csv")
# COLLECTION_03_CHUNKS        = load_chunks_csv("processed_chunks/COLLECTION_03_CHUNKS.csv")
# COLLECTION_04_CHUNKS            = load_chunks_csv("processed_chunks/COLLECTION_04_CHUNKS.csv")
# COLLECTION_05_CHUNKS                  = load_chunks_csv("processed_chunks/COLLECTION_05_CHUNKS.csv")
# COLLECTION_06_CHUNKS                = load_chunks_csv("processed_chunks/COLLECTION_06_CHUNKS.csv")
COLLECTION_07_CHUNKS            = load_chunks_csv("processed_chunks/COLLECTION_07_CHUNKS.csv")
# COLLECTION_08_CHUNKS   = load_chunks_csv("processed_chunks/COLLECTION_08_CHUNKS.csv")
# COLLECTION_09_CHUNKS         = load_chunks_csv("processed_chunks/COLLECTION_09_CHUNKS.csv")
# COLLECTION_10_CHUNKS   = load_chunks_csv("processed_chunks/COLLECTION_10_CHUNKS.csv")
# COLLECTION_11_CHUNKS  = load_chunks_csv("processed_chunks/COLLECTION_11_CHUNKS.csv")


# --- Embed and store all ---
# embed_and_store_chunks(COLLECTION_01_CHUNKS,               collections[COLLECTION_01],                    COLLECTION_01                )
# embed_and_store_chunks(COLLECTION_02_CHUNKS,            collections[COLLECTION_02],              COLLECTION_02          )
# embed_and_store_chunks(COLLECTION_03_CHUNKS,       collections[COLLECTION_03],         COLLECTION_03     )
# embed_and_store_chunks(COLLECTION_04_CHUNKS,           collections[COLLECTION_04],              COLLECTION_04          )
# embed_and_store_chunks(COLLECTION_05_CHUNKS,                 collections[COLLECTION_05],                   COLLECTION_05               )
# embed_and_store_chunks(COLLECTION_06_CHUNKS,               collections[COLLECTION_06],                 COLLECTION_06             )
embed_and_store_chunks(COLLECTION_07_CHUNKS,           collections[COLLECTION_07],             COLLECTION_07         )
# embed_and_store_chunks(COLLECTION_08_CHUNKS,  collections[COLLECTION_08],     COLLECTION_08 )
# embed_and_store_chunks(COLLECTION_09_CHUNKS,        collections[COLLECTION_09],           COLLECTION_09       )
# embed_and_store_chunks(COLLECTION_10_CHUNKS,    collections[COLLECTION_10],       COLLECTION_10   )
# embed_and_store_chunks(COLLECTION_11_CHUNKS, collections[COLLECTION_11],    COLLECTION_11)

print("\nAll collections stored successfully.")
# print(f"COLLECTION 01:                    {collections[COLLECTION_01].count()} vectors")
# print(f"COLLECTION 02:              {collections[COLLECTION_02]].count()} vectors")
# print(f"COLLECTION 03:         {collections[COLLECTION_03].count()} vectors")
# print(f"COLLECTION 014:              {collections[COLLECTION_04].count()} vectors")
# print(f"COLLECTION 05:                   {collections[COLLECTION_05].count()} vectors")
# print(f"COLLECTION 06:                 {collections[COLLECTION_06].count()} vectors")
print(f"COLLECTION 07:             {collections[COLLECTION_07].count()} vectors")
# print(f"COLLECTION 08:     {collections[COLLECTION_08].count()} vectors")
# print(f"COLLECTION 09:           {collections[COLLECTION_09].count()} vectors")
# print(f"COLLECTION 10:     {collections[COLLECTION_10].count()} vectors")
# print(f"COLLECTION 11:    {collections[COLLECTION_11].count()} vectors")

In [ ]:
import os

def get_dir_size_mb(path):
    total = 0
    for dirpath, _, filenames in os.walk(path):
        for f in filenames:
            total += os.path.getsize(os.path.join(dirpath, f))
    return total / (1024 ** 2)

print(f"{get_dir_size_mb('./chroma_db'):.2f} MB")

### After creating ChromaDB

In [ ]:
client = chromadb.PersistentClient(path="./chroma_db")

collections = {
    name: client.get_or_create_collection(name=name)
    for name in [
        COLLECTION_01, COLLECTION_02, COLLECTION_03, COLLECTION_04,
        COLLECTION_05, COLLECTION_06, COLLECTION_07,
        COLLECTION_08, COLLECTION_09,
        COLLECTION_10, COLLECTION_11
    ]
}

# Verify
for name, collection in collections.items():
    print(f"{name}: {collection.count()} entries")

In [ ]:
def show_sample_entry(collection_name: str, n: int = 1):
    collection = collections[collection_name]
    result = collection.get(limit=n, include=["metadatas", "documents"])

    for i in range(len(result["ids"])):
        print(f"=== {collection_name} | entry {i+1} ===")
        print(f"ID:       {result['ids'][i]}")
        print(f"Metadata: {result['metadatas'][i]}")
        print(f"Text:     {result['documents'][i]}")
        print()

# Example
show_sample_entry(COLLECTION_01)
show_sample_entry(COLLECTION_02)
show_sample_entry(COLLECTION_03)
show_sample_entry(COLLECTION_04)
show_sample_entry(COLLECTION_05)
show_sample_entry(COLLECTION_06)
show_sample_entry(COLLECTION_07)
show_sample_entry(COLLECTION_08)
show_sample_entry(COLLECTION_11)
show_sample_entry(COLLECTION_09)
show_sample_entry(COLLECTION_10)

In [ ]:
import pandas as pd
import json

# For EPD — aggregate all chunks per product into one cell
epd_df = pd.read_csv("processed_chunks/COLLECTION_01_CHUNKS.csv")
epd_df["product_id"] = epd_df["metadata"].apply(
    lambda x: str(json.loads(x).get("product_id", ""))
)
epd_grouped = (
    epd_df.groupby("product_id")["text"]
    .apply(lambda texts: "\n---\n".join(texts))
    .reset_index()
    .rename(columns={"text": COLLECTION_01})
)

def load_and_combine_chunks(chunk_files: dict, preloaded: dict = None) -> pd.DataFrame:
    """
    chunk_files: {chunk_type: filepath}
    preloaded:   {chunk_type: dataframe} — skip CSV loading for these
    """
    all_dfs = []
    preloaded = preloaded or {}

    for chunk_type, filepath in chunk_files.items():
        # Use preloaded df if available
        if chunk_type in preloaded:
            df = preloaded[chunk_type][["product_id", chunk_type]]
            all_dfs.append(df)
            print(f"Using preloaded {chunk_type}: {len(df)} rows")
            continue

        try:
            df = pd.read_csv(filepath)
            df["product_id"] = df["metadata"].apply(
                lambda x: str(json.loads(x).get("product_id", ""))
            )
            df = df.rename(columns={"text": chunk_type})[["product_id", chunk_type]]
            all_dfs.append(df)
            print(f"Loaded {len(df)} rows from {filepath}")
        except Exception as e:
            print(f"Failed to load {filepath}: {e}")

    combined = all_dfs[0]
    for df in all_dfs[1:]:
        combined = combined.merge(df, on="product_id", how="outer")

    print(f"\nCombined: {len(combined)} unique products")
    return combined


CHUNK_FILES = {
    COLLECTION_01:                 "processed_chunks/COLLECTION_01_CHUNKS.csv",   # will be overridden by preloaded
    COLLECTION_02:           "processed_chunks/COLLECTION_02_CHUNKS.csv",
    COLLECTION_03:      "processed_chunks/COLLECTION_03_CHUNKS.csv",
    COLLECTION_04:           "processed_chunks/COLLECTION_04_CHUNKS.csv",
    COLLECTION_05:                "processed_chunks/COLLECTION_05_CHUNKS.csv",
    COLLECTION_06:              "processed_chunks/COLLECTION_06_CHUNKS.csv",
    COLLECTION_07:          "processed_chunks/COLLECTION_07_CHUNKS.csv",
    COLLECTION_08:  "processed_chunks/COLLECTION_08_CHUNKS.csv",
    COLLECTION_09:        "processed_chunks/COLLECTION_09_CHUNKS.csv",
    COLLECTION_10:  "processed_chunks/COLLECTION_10_CHUNKS.csv",
    COLLECTION_11: "processed_chunks/COLLECTION_11_CHUNKS.csv",
}

combined_df = load_and_combine_chunks(
    CHUNK_FILES,
    preloaded={COLLECTION_01: epd_grouped}  # pass grouped EPD here
)
combined_df.to_csv("processed_chunks/combined_chunks.csv", index=False)
print("Saved to processed_chunks/combined_chunks.csv")